In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ellipticco/elliptic-data-set")

print("Path to dataset files:", path)

import torch

TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = "cpu"  # Colab handles GPU automatically

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html
!pip install -q torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html
!pip install -q torch-spline-conv -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html
!pip install -q torch-geometric

class Config:
    def __init__(self):
        # Dataset paths
        self.features_path = "features.csv"
        self.edges_path = "edgelist.csv"
        self.classes = "classes.csv"

        # Feature dimension (auto-detect recommended)
        self.input_dim = 165   # Elliptic has 165 features

        # Model size (moderate power)
        self.hidden_dim = 128

        # 2 classes: licit, illicit (unknown is ignored in loss calculation)
        self.output_dim = 2

        # Regularization (important for Elliptic)
        self.dropout = 0.5

        # Optimization
        self.lr = 0.001
        self.weight_decay = 5e-4

        # Training length
        self.epochs = 150

# Instantiate the config object
config = Config()

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler


class EllipticDataset:
    def __init__(self, config):
        self.features_df = pd.read_csv(config.features_path, header=None)
        self.edges_df = pd.read_csv(config.edges_path)
        self.labels_df = pd.read_csv(config.classes)

        # Map classes
        # licit = 0, illicit = 1, unknown = 2
        self.labels_df["class"] = self.labels_df["class"].map({'unknown': 2, '1': 1, '2': 0})

        self.merged_df = self.merge()

        self.edge_index = self._edge_index()
        self.edge_weights = self._edge_weights()
        self.node_features = self._node_features()
        self.labels = self._labels()

        self.train_mask, self.val_mask, self.test_mask = self._create_masks()

    # ------------------------------------------------

    def merge(self):
        df_merge = self.features_df.merge(
            self.labels_df, how='left', right_on="txId", left_on=0
        )
        df_merge = df_merge.sort_values(0).reset_index(drop=True)
        return df_merge

    # ------------------------------------------------

    def _node_features(self):
        node_features = self.merged_df.drop(['txId'], axis=1).copy()
        node_features = node_features.drop(columns=[0, 1, "class"])

        # 🔥 Feature Normalization (VERY IMPORTANT)
        scaler = StandardScaler()
        node_features = scaler.fit_transform(node_features.values)

        return torch.tensor(node_features, dtype=torch.float)

    # ------------------------------------------------

    def _labels(self):
        labels = self.merged_df["class"].values
        return torch.tensor(labels, dtype=torch.long)

    # ------------------------------------------------

    def _edge_index(self):
        node_ids = self.merged_df[0].values
        ids_mapping = {y: x for x, y in enumerate(node_ids)}

        edges = self.edges_df.copy()
        edges.txId1 = edges.txId1.map(ids_mapping)
        edges.txId2 = edges.txId2.map(ids_mapping)
        edges = edges.astype(int)

        edge_index = torch.tensor(edges.values.T, dtype=torch.long)

        # 🔥 Make graph undirected
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

        return edge_index.contiguous()

    # ------------------------------------------------

    def _edge_weights(self):
        return torch.ones(self.edge_index.shape[1], dtype=torch.float)

    # ------------------------------------------------

    def _create_masks(self):
        labels = self.labels.numpy()
        timesteps = self.merged_df[1].values  # timestep column

        train_mask = torch.zeros(len(labels), dtype=torch.bool)
        val_mask = torch.zeros(len(labels), dtype=torch.bool)
        test_mask = torch.zeros(len(labels), dtype=torch.bool)

        # Only classified nodes (ignore unknown)
        classified = labels != 2

        # 🔥 Temporal split (proper for Elliptic)
        train_mask[(timesteps < 35) & classified] = True
        val_mask[(timesteps >= 35) & (timesteps < 40) & classified] = True
        test_mask[(timesteps >= 40) & classified] = True

        return train_mask, val_mask, test_mask

    # ------------------------------------------------

    def get_class_weights(self):
        labels = self.labels[self.train_mask]
        class_sample_count = torch.bincount(labels[labels != 2])
        weight = 1. / class_sample_count.float()
        return weight

    # ------------------------------------------------

    def pyg_dataset(self):
        data = Data(
            x=self.node_features,
            edge_index=self.edge_index,
            edge_attr=self.edge_weights,
            y=self.labels,
            train_mask=self.train_mask,
            val_mask=self.val_mask,
            test_mask=self.test_mask
        )
        return data


!mv elliptic_txs_features.csv features.csv
!mv elliptic_txs_classes.csv classes.csv
!mv elliptic_txs_edgelist.csv edgelist.csv

!ls features.csv
!ls edgelist.csv
!ls classes.csv

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv


class GraphSAGE(nn.Module):
    def __init__(self, config):
        super(GraphSAGE, self).__init__()

        self.input_dim = config.input_dim
        self.hidden_dim = config.hidden_dim
        self.output_dim = config.output_dim
        self.dropout = config.dropout

        # GraphSAGE layers
        self.conv1 = SAGEConv(self.input_dim, self.hidden_dim)
        self.bn1 = nn.BatchNorm1d(self.hidden_dim)

        self.conv2 = SAGEConv(self.hidden_dim, self.hidden_dim)
        self.bn2 = nn.BatchNorm1d(self.hidden_dim)

        # Classifier
        self.classifier = nn.Linear(self.hidden_dim, self.output_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # First layer
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Second layer
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Output logits (NO sigmoid)
        x = self.classifier(x)

        return x


import sys
sys.path.append("/content")

!mv models.py model.py
!mv datasets.py dataset.py


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


import torch
import torch.nn.functional as F

# The `device`, `EllipticDataset`, `GraphSAGE`, `config` objects are already defined or imported in other cells.
# They will be accessible here because they are in the global scope of the notebook.

class Trainer:
    def __init__(self, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load dataset
        self.dataset = EllipticDataset(config)
        self.data = self.dataset.pyg_dataset().to(self.device)

        # Initialize model
        self.model = GraphSAGE(config).to(self.device)

        # Class weights (handle class imbalance)
        self.class_weights = self.dataset.get_class_weights().to(self.device)

        # Loss function
        self.criterion = torch.nn.CrossEntropyLoss(weight=self.class_weights)

        # Optimizer
        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=config.lr,
            weight_decay=config.weight_decay
        )

    def train_epoch(self):
        self.model.train()
        self.optimizer.zero_grad()

        out = self.model(self.data)  # raw logits
        loss = self.criterion(out[self.data.train_mask], self.data.y[self.data.train_mask])

        loss.backward()
        self.optimizer.step()

        return loss.item()
    @torch.no_grad()
    def evaluate(self, mask):
        self.model.eval()
        out = self.model(self.data)

        pred = out.argmax(dim=1)

        y_true = self.data.y[mask].cpu().numpy()
        y_pred = pred[mask].cpu().numpy()

        acc = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=0)

        # 🔥 Illicit Recall (class = 1)
        illicit_recall = recall_score(y_true, y_pred, labels=[1], average='macro', zero_division=0)

        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

        return acc, precision, recall, illicit_recall, f1

    def run_training(self):
        best_val_acc = 0
        best_test_acc = 0

        print("Starting training...")
        for epoch in range(self.config.epochs):
            loss = self.train_epoch()
            train_acc, train_prec, train_rec, train_illicit_rec, train_f1 = self.evaluate(self.data.train_mask)
            val_acc, val_prec, val_rec, val_illicit_rec, val_f1 = self.evaluate(self.data.val_mask)
            test_acc, test_prec, test_rec, test_illicit_rec, test_f1 = self.evaluate(self.data.test_mask)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_test_acc = test_acc

            if epoch % 10 == 0:
                print(f"Epoch {epoch:03d} | Loss: {loss:.4f}")

                print(f"Train → Acc:{train_acc:.3f} | Prec:{train_prec:.3f} | "
                  f"Rec:{train_rec:.3f} | IllicitRec:{train_illicit_rec:.3f} | F1:{train_f1:.3f}")

                print(f"Val   → Acc:{val_acc:.3f} | Prec:{val_prec:.3f} | "
                  f"Rec:{val_rec:.3f} | IllicitRec:{val_illicit_rec:.3f} | F1:{val_f1:.3f}")

                print(f"Test  → Acc:{test_acc:.3f} | Prec:{test_prec:.3f} | "
                  f"Rec:{test_rec:.3f} | IllicitRec:{test_illicit_rec:.3f} | F1:{test_f1:.3f}")


import os

class Config:
    def __init__(self):

        # ---------------------------
        # Dataset paths
        # ---------------------------
        global path # Access the 'path' variable from the previously executed cell
        # The files are actually in a subfolder named 'elliptic_bitcoin_dataset'
        data_subpath = os.path.join(path, "elliptic_bitcoin_dataset")

        self.features_path = os.path.join(data_subpath, "elliptic_txs_features.csv")
        self.edges_path = os.path.join(data_subpath, "elliptic_txs_edgelist.csv")
        self.classes = os.path.join(data_subpath, "elliptic_txs_classes.csv")

        # ---------------------------
        # Model hyperparameters
        # ---------------------------
        self.hidden_dim = 128      # Moderate capacity
        self.output_dim = 2        # licit, illicit (unknown is ignored in loss calculation)
        self.dropout = 0.5

        # ---------------------------
        # Optimization
        # ---------------------------
        self.lr = 0.001
        self.weight_decay = 5e-4
        self.epochs = 150

        # input_dim will be set after dataset loads
        self.input_dim = 165 # Elliptic has 165 features

import os
print(f"Contents of {path}:")
!ls -F {path}

config = Config()

trainer = Trainer(config)

trainer.run_training()

Using Colab cache for faster access to the 'elliptic-data-set' dataset.
Path to dataset files: /kaggle/input/elliptic-data-set
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.4/682.4 kB 41.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.2/828.2 kB 50.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.9/306.9 kB 22.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.1 MB/s eta 0:00:00
mv: cannot stat 'elliptic_txs_features.csv': No such file or directory
mv: cannot stat 'elliptic_txs_classes.csv': No such file or directory
mv: cannot stat 'elliptic_txs_edgelist.csv': No such file or directory
ls: cannot access 'features.csv': No such file or directory
ls: cannot access 'edgelist.csv': No such file or directory
ls: cannot access 'classes.csv': No such file or directory
mv: